In [110]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [1]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

import torch
import torch_geometric as pyg

from baseline_evals.feature_selection import variance_filtering

from torch_geometric.utils import to_networkx


from bipartite_gnn.train_test import GNNTrainer
from bipartite_gnn.model import GAT_2L, BipartiteRGAT, BiRGAT
from bipartite_gnn.graph_building import cosine_similarity_matrix, dense_to_coo
from bipartite_gnn.preprocessing import gg_interactions, get_mirna_gene_interactions, pp_interactions

In [2]:
# Check currently allocated CUDA memory in bytes
allocated_memory = torch.cuda.memory_allocated()
print(f"Currently allocated CUDA memory: {allocated_memory} bytes")

# Check maximum allocated CUDA memory in bytes
max_allocated_memory = torch.cuda.max_memory_allocated()
print(f"Maximum allocated CUDA memory: {max_allocated_memory} bytes")


Currently allocated CUDA memory: 0 bytes
Maximum allocated CUDA memory: 0 bytes


In [3]:
from calendar import c


null_vals = ["NA"]
mrna = pl.read_csv("BRCA_PROCESSED_DATA/mrna.tsv", separator="\t", null_values=null_vals)
cnv = pl.read_csv("BRCA_PROCESSED_DATA/cnv.tsv", separator="\t", null_values=null_vals)
cna_th = pl.read_csv("BRCA_PROCESSED_DATA/cnvth.tsv", separator="\t", null_values=null_vals)
mirna = pl.read_csv("BRCA_PROCESSED_DATA/mirna.tsv", separator="\t", null_values=null_vals)

labels = pl.read_csv("BRCA_PROCESSED_DATA/labels.tsv", separator="\t")
le = LabelEncoder()
le.fit(labels["PAM50_mRNA_nature2012"].to_list())
y = le.transform(labels["PAM50_mRNA_nature2012"].to_list())
# labels, y

In [4]:
# mrna, cna, mirna
mrna_gene_names = mrna[:, 0].to_list()
cna_gene_names = cna[:, 0].to_list()
mirna_gene_names = mirna[:, 0].to_list()

In [5]:
mirna_gene_names = pl.read_csv("miRBaseConverter_Result.csv")
mirna_targets = mirna_gene_names["TargetName"].to_list()

In [6]:
mrna_X = torch.tensor(mrna[:, 1:].to_numpy().T)
cna_X = torch.tensor(cna[:, 1:].to_numpy().T, dtype=torch.float32)
mirna_X = torch.tensor(mirna[:, 1:].to_numpy().T)

In [7]:
train_mask, test_mask = train_test_split(torch.arange(len(y)), test_size=0.4, stratify=y, random_state=5)
val_mask, test_mask = train_test_split(test_mask, test_size=0.5, random_state=5, stratify=y[test_mask])

In [120]:
from bipartite_gnn.graph_building import create_diff_exp_connections_nbnom, create_diff_exp_connections_norm, dense_to_coo
from baseline_evals.feature_selection import class_variational_selection


# select 400 top genes for mrna
# select_mask = variance_filtering(mrna_X[train_mask].numpy(), 5000)
select_mask_mrna, _ = class_variational_selection(mrna_X[train_mask], y[train_mask], 3000)
select_mask_cna, _ = class_variational_selection(cna[train_mask], y[train_mask], 3000)

mrna_X_400 = mrna_X[:, select_mask]

mrna_genes = np.array(mrna_gene_names)[select_mask]
# select 400 top genes for cna
# select_mask = variance_filtering(cna_X.numpy(), 500)
# cna_X_400 = cna_X[:, select_mask].float()
 
mrna_A = create_diff_exp_connections_norm(mrna_X_400, train_mask, 2.5)
# mrna_A_n = create_diff_exp_connections_nbnom(mrna_X_400, train_mask, 2.5)
# mrna_A = torch.logical_or(mrna_A, mrna_A_n).int()

print(mrna_A.shape)

connected_nodes_mask = mrna_A.sum(dim=0) != 0

mrna_A = mrna_A[:, connected_nodes_mask]
mrna_X_400 = mrna_X_400[:, connected_nodes_mask]

mrna_genes = mrna_genes[connected_nodes_mask]

print(mrna_A.shape, mrna_X_400.shape)

print("isolated sample nodes, isolated gene nodes, mean degree: ")
print(
    (mrna_A.abs().sum(axis=1) == 0).sum(),
    (mrna_A.abs().sum(axis=0) == 0).sum(),
    mrna_A.abs().sum() / mrna_A.shape[0],
)

# cna_A = create_diff_exp_connections_norm(cna_X_400, train_mask, 1.0)

mirna_A = create_diff_exp_connections_norm(mirna_X, train_mask, 2.0)

# normalize mrna_X_400
std_scaler = StandardScaler()
std_scaler.fit(mrna_X_400.numpy()[train_mask])

mrna_X_400 = torch.tensor(std_scaler.transform(mrna_X_400.numpy()))

# # # normalize cna_X
# # std_scaler = StandardScaler()
# # std_scaler.fit(cna_X.numpy()[train_mask])

# # cna_X = torch.tensor(std_scaler.transform(cna_X.numpy()))

# normalize mirna_X
std_scaler = StandardScaler()
std_scaler.fit(mirna_X.numpy()[train_mask])

mirna_X = torch.tensor(std_scaler.transform(mirna_X.numpy()))

isolated sample nodes, isolated gene nodes, mean degree: 
tensor(0) tensor(50) tensor(56.3851, dtype=torch.float64)
torch.Size([483, 3000])
torch.Size([483, 2774]) torch.Size([483, 2774])
isolated sample nodes, isolated gene nodes, mean degree: 
tensor(0) tensor(0) tensor(53.6439, dtype=torch.float64)
isolated sample nodes, isolated gene nodes, mean degree: 
tensor(35) tensor(0) tensor(10.5818, dtype=torch.float64)


In [121]:
gg_A = gg_interactions(mrna_genes, mrna_genes)

pp_A = pp_interactions(mrna_genes, mrna_genes)

mrna_features_A = torch.logical_or(gg_A, pp_A).int()

mrna_features_A.sum()

tensor(25785)

In [122]:
mirna_mrna_interactions = get_mirna_gene_interactions(mirna_targets, mrna_genes)

torch.Size([231, 2774])


In [123]:
# create hetero-data object
import torch_geometric.transforms as T

from bipartite_gnn.graph_building import dense_to_attributes

data = pyg.data.HeteroData()

proj_dim = 256

# sample node features
data["mrna"].x = mrna_X_400.float()
# data["cna"].x = cna_X_400.float()
# data["mirna"].x = mirna_X.float()

data.y = torch.tensor(y)

data.omics = ["mrna"] #, "mirna"] #, "cna", "mirna"]
data.feature_names = ["mrna_feature", "mirna_feature"] #, "cna_feature", "mirna_feature"]

# feature node projection
data["mrna_feature"].x = torch.zeros(mrna_X_400.shape[1], proj_dim)
# data["cna_feature"].x = torch.zeros(cna_X_400.shape[1], proj_dim)
# data["mirna_feature"].x = torch.zeros(mirna_X.shape[1], proj_dim)

# create edges
 # 3 for forward egdes, 3 for backward edges

data["mrna", "diff_exp", "mrna_feature"].edge_index = dense_to_coo(mrna_A)
data["mrna", "diff_exp", "mrna_feature"].edge_attributes = dense_to_attributes(mrna_A)
data["mrna_feature", "interaction", "mrna_feature"].edge_index = dense_to_coo(mrna_features_A)

# data["mirna", "diff_exp", "mirna_feature"].edge_index = dense_to_coo(mirna_A)
# data["mirna", "diff_exp", "mirna_feature"].edge_attributes = dense_to_attributes(mirna_A)

# data["mirna_feature", "interacts", "mrna_feature"].edge_index = dense_to_coo(mirna_mrna_interactions)

# data["mrna", "self_loop", "mrna"].edge_index = dense_to_coo(torch.eye(mrna_X_400.shape[0]))
# data["mrna_feature", "self_loop", "mrna_feature"].edge_index = dense_to_coo(torch.eye(mrna_X_400.shape[1]))
# data["cna", "diff_exp", "cna_feature"].edge_index = dense_to_coo(cna_A)
# data["mirna", "diff_exp", "mirna_feature"].edge_index = dense_to_coo(mirna_A)

data = T.ToUndirected()(data)
# data = T.AddSelfLoops()(data)

# 2 for forward and backward diff exp, one for each self loop
data.num_relations = len(data.edge_index_dict.keys())
print("num_relations", data.num_relations)

data["train_mask"] = train_mask
data["val_mask"] = val_mask
data["test_mask"] = test_mask

data

num_relations 3


HeteroData(
  y=[483],
  omics=[1],
  feature_names=[2],
  num_relations=3,
  train_mask=[289],
  val_mask=[97],
  test_mask=[97],
  mrna={ x=[483, 2774] },
  mrna_feature={ x=[2774, 256] },
  (mrna, diff_exp, mrna_feature)={
    edge_index=[2, 25910],
    edge_attributes=[25910, 1],
  },
  (mrna_feature, interaction, mrna_feature)={ edge_index=[2, 41291] },
  (mrna_feature, rev_diff_exp, mrna)={
    edge_index=[2, 25910],
    edge_attributes=[25910, 1],
  }
)

In [ ]:
from baseline_evals.mlp_eval import mlp_eval

mlp_eval(mrna_X_400, y, n_features=3000)

In [127]:
# model = BipartiteRGAT(
#     input_sizes=[mrna_X_400.shape[1]], # , cna_X_400.shape[1], mirna_X_200.shape[1]],
#     proj_dim=proj_dim,
#     hidden_channels=[128, 64], # num_layers = len(hidden_channels) - 1
#     num_labels=len(np.unique(y)),
#     num_relations=data.num_relations,
#     num_heads=1,
#     feature_integration_mode="linear",
# )

model = BiRGAT(
    omic_channels=data.omics,
    feature_names=data.feature_names,
    relations=list(data.edge_index_dict.keys()),
    input_dims=[mrna_X_400.shape[1]],# mirna_X.shape[1]], # , cna_X_400.shape[1], mirna_X.shape[1]],
    num_classes=len(np.unique(y)),
    proj_dim=proj_dim,
    hidden_channels=[256, 128, 128, 128],
    heads=4,
    dropout=0.4,
)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=5e-3)

ccounts = torch.unique(data.y[train_mask], return_counts=True)[1]
inv_w = 1.0 / ccounts.float()
class_weights = inv_w / inv_w.sum()

trainer = GNNTrainer(
    model=model,
    loss_fn=torch.nn.CrossEntropyLoss(weight=class_weights),
    optimizer=optimizer,
    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=15, min_lr=1e-5),
    params={
        "l1_lambda" : 0.01,
    }
)

trainer.train(
    data=data,
    epochs=150,
    log_interval=5,
)

Epoch: 005, 
Train Loss: 68.7163, Train Acc: 0.3737, Train F1 Macro: 0.3930, Train F1 Weighted: 0.3879
Val Loss: 1.3218, Val Acc: 0.4742, Val F1 Macro: 0.5140, Val F1 Weighted: 0.3786
Test Loss: 1.3261, Test Acc: 0.4433, Test F1 Macro: 0.4819, Test F1 Weighted: 0.3562
##################################################
Epoch: 010, 
Train Loss: 68.6032, Train Acc: 0.5709, Train F1 Macro: 0.5449, Train F1 Weighted: 0.5666
Val Loss: 1.1878, Val Acc: 0.6495, Val F1 Macro: 0.6546, Val F1 Weighted: 0.6505
Test Loss: 1.1979, Test Acc: 0.6804, Test F1 Macro: 0.7072, Test F1 Weighted: 0.6836
##################################################
Epoch: 015, 
Train Loss: 68.4000, Train Acc: 0.5640, Train F1 Macro: 0.5448, Train F1 Weighted: 0.5601
Val Loss: 0.9718, Val Acc: 0.6907, Val F1 Macro: 0.6672, Val F1 Weighted: 0.6887
Test Loss: 0.9789, Test Acc: 0.6907, Test F1 Macro: 0.6459, Test F1 Weighted: 0.6953
##################################################
Epoch: 020, 
Train Loss: 68.2691, Train 

KeyboardInterrupt: 